In [ ]:
import os
import shutil
import sys

# Tell PySpark where the Windows translator files live
os.environ['HADOOP_HOME'] = 'C:\\hadoop'

# Make PySpark use the same Python executable as this notebook kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Notebook Python: {sys.executable}")

# Check Java before importing/starting Spark, so the notebook fails clearly instead of hanging
if not os.environ.get("JAVA_HOME") and shutil.which("java") is None:
    raise RuntimeError("PySpark needs Java before SparkSession can start. Install a JDK and set JAVA_HOME, then restart VS Code/Jupyter.")

print("PySpark setup check passed.")

Notebook Python: c:\Users\zraym\AppData\Local\Programs\Python\Python314\python.exe
PySpark setup check passed.


In [5]:
print("About to import SparkSession...", flush=True)
from pyspark.sql import SparkSession
print("SparkSession imported.", flush=True)

from pyspark.sql.functions import monotonically_increasing_id, col, when
print("PySpark SQL functions imported.", flush=True)

# start up the spark engine
print("About to start Spark engine...", flush=True)
spark = SparkSession.builder \
    .appName("SECOM_Data_Engine") \
    .master("local[1]") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

print("Spark Engine Online!", flush=True)

# raw files ingestion
# inferSchema = True to automatically figure out if a column is a float, int, or string
sensors_df = spark.read.csv('../data/raw/secom.data', sep=' ', header=False, inferSchema=True)
labels_df = spark.read.csv('../data/raw/secom_labels.data', sep=' ', header=False, inferSchema=True)

# rename the label colums because pysparks 
# defaults to _c0, _c1, etc
labels_df = labels_df.withColumnRenamed("_c0", "Result").withColumnRenamed("_c1", "Timestamp")

# changing all the -1 pass to 0 just like before
labels_df = labels_df.withColumn("Result", when(col("Result") == -1, 0).otherwise(col("Result")))

# assigning a id number to every row, saved in row_id for both sensors and labels
sensors_df = sensors_df.withColumn("row_id", monotonically_increasing_id())
labels_df = labels_df.withColumn("row_id", monotonically_increasing_id())

# matching each id with the other id, sipping them together
# and then deleting the row id as it is not useful anymore
df_spark = labels_df.join(sensors_df, on="row_id", how="inner").drop("row_id")

# pyspark didnt actually do anything yet it jsut writes down instructions to do it. 
row_count = df_spark.count() # action command, execute all the stitching instructions
col_count = len(df_spark.columns)
print(f"\nDistributed Dataset Shape: ({row_count}, {col_count})")

# look at the data
df_spark.select("Result", "Timestamp", "_c0", "_c1", "_c2").show(5)

KeyboardInterrupt: 